# FitLog Agent: Midterm (41-601 Agentic AI)

A naive tool-calling agent that answers questions about a personal workout log,
using Mango 1.5 9B via the Orion API.

**Structure**
1. Setup
2. Data
3. Tools (4 tools, each = Python function + JSON schema)
4. Agent loop
5. Test queries, ground truth and diagnosis

## 1. Setup

Requirements: `pip install -r requirements.txt`, and an `ORION_API_KEY` in a `.env` file (see `.env.example`).

In [ ]:
# Install the requirements (uncomment to run once):
# %pip install -r requirements.txt

In [4]:
import os, json, math
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(".env")
ORION_BASE_URL = "https://api.orion.cmkl.ai/v1"
ORION_API_KEY  = os.getenv("ORION_API_KEY")
ORION_MODEL    = "Mango 1.5-9B"

DATA_DIR = Path("data")

print("API key found:", bool(ORION_API_KEY))

API key found: True


## 2. Data

Mock data for one example user (3 weeks, 2-22 Nov 2026):
- `workout_log.csv`: every logged exercise (gym) and badminton session (minutes)
- `exercises.csv`: reference table with muscle groups, equipment and how weight is recorded
- `guides/`: 6 short training-rule documents, used by the RAG tool

In [9]:
WORKOUT_LOG = pd.read_csv(DATA_DIR / "workout_log.csv",
                          dtype={"sets": "Int64", "reps": "Int64", "duration_min": "Int64"})
EXERCISES   = pd.read_csv(DATA_DIR / "exercises.csv")

print(f"Workout log: {len(WORKOUT_LOG)} rows, {WORKOUT_LOG['date'].min()} to {WORKOUT_LOG['date'].max()}")
print(f"Exercises:   {len(EXERCISES)} rows")
display(WORKOUT_LOG.head(8))
display(EXERCISES.head(8))

Workout log: 55 rows, 2026-11-02 to 2026-11-22
Exercises:   20 rows


,date,day,session,exercise,sets,reps,weight_kg,duration_min,notes
0,2026-11-02,Mon,Badminton,Badminton,<NA>,<NA>,NaN,180,doubles games
1,2026-11-03,Tue,Upper A,Bench Press,3,8,60.0,<NA>,NaN
2,2026-11-03,Tue,Upper A,Barbell Row,3,8,50.0,<NA>,NaN
3,2026-11-03,Tue,Upper A,Overhead Press,3,8,35.0,<NA>,NaN
4,2026-11-03,Tue,Upper A,Lat Pulldown,3,10,45.0,<NA>,NaN
5,2026-11-03,Tue,Upper A,Bicep Curl,3,12,10.0,<NA>,NaN
6,2026-11-04,Wed,Badminton,Badminton,<NA>,<NA>,NaN,180,doubles games
7,2026-11-05,Thu,Lower A,Back Squat,4,6,70.0,<NA>,NaN


,name,primary_muscle,secondary_muscles,equipment,load_note
0,Bench Press,chest,triceps;shoulders,barbell,total bar weight
1,Incline Dumbbell Press,chest,shoulders;triceps,dumbbell,weight per dumbbell
2,Barbell Row,back,biceps,barbell,total bar weight
3,Lat Pulldown,back,biceps,cable machine,machine stack weight
4,Pull-Up,back,biceps,bodyweight,0 = bodyweight only
5,Seated Cable Row,back,biceps,cable machine,machine stack weight
6,Overhead Press,shoulders,triceps,barbell,total bar weight
7,Lateral Raise,shoulders,NaN,dumbbell,weight per dumbbell


## 3. Tools

Each tool has a Python function and a JSON schema. The model reads the schema to decide when and how
to call a tool. It only outputs the tool name and arguments; the agent loop executes the function.

### Tool 1: `get_workouts` - logged workouts by date

Inputs: `start_date`, `end_date` (YYYY-MM-DD) and optional `exercise`.
Dates are compared as text, which works because `YYYY-MM-DD` sorts correctly.

In [4]:
def get_workouts(start_date: str, end_date: str, exercise: str = None) -> str:
    """Return all logged workout rows between two dates (inclusive)."""
    rows = WORKOUT_LOG[(WORKOUT_LOG["date"] >= start_date) & (WORKOUT_LOG["date"] <= end_date)]
    if exercise:
        rows = rows[rows["exercise"].str.lower().str.contains(exercise.lower())]
    if rows.empty:
        return json.dumps({"result": "No workouts found", "start_date": start_date,
                           "end_date": end_date, "exercise": exercise})
    return rows.to_json(orient="records")


get_workouts_schema = {
    "type": "function",
    "function": {
        "name": "get_workouts",
        "description": "Get the user's logged workouts between two dates. Each row has date, session, "
                       "exercise, sets, reps, weight_kg, duration_min (badminton) and notes. "
                       "Optionally filter by exercise name.",
        "parameters": {
            "type": "object",
            "properties": {
                "start_date": {"type": "string", "description": "Start date in YYYY-MM-DD format"},
                "end_date":   {"type": "string", "description": "End date in YYYY-MM-DD format (inclusive)"},
                "exercise":   {"type": "string", "description": "Optional exercise name, e.g. 'Bench Press'"},
            },
            "required": ["start_date", "end_date"],
        },
    },
}

In [5]:
# Example: all bench press sessions in the log
print(get_workouts("2026-11-02", "2026-11-22", exercise="bench"))

[{"date":"2026-11-03","day":"Tue","session":"Upper A","exercise":"Bench Press","sets":3,"reps":8,"weight_kg":60.0,"duration_min":null,"notes":null},{"date":"2026-11-10","day":"Tue","session":"Upper A","exercise":"Bench Press","sets":3,"reps":8,"weight_kg":62.5,"duration_min":null,"notes":null},{"date":"2026-11-17","day":"Tue","session":"Upper A","exercise":"Bench Press","sets":3,"reps":8,"weight_kg":65.0,"duration_min":null,"notes":null}]


### Tool 2: `get_exercise_info` - exercise lookup

Returns primary muscle, secondary muscles, equipment, and `load_note`
(e.g. dumbbell weight is recorded per dumbbell).

In [6]:
def get_exercise_info(name: str) -> str:
    """Return the muscle groups, equipment and load note for an exercise."""
    match = EXERCISES[EXERCISES["name"].str.lower().str.contains(name.lower())]
    if match.empty:
        return json.dumps({"result": "Exercise not found", "name": name})
    return match.fillna("").to_json(orient="records")


get_exercise_info_schema = {
    "type": "function",
    "function": {
        "name": "get_exercise_info",
        "description": "Look up an exercise to find its primary muscle group, secondary muscles, "
                       "equipment, and how its weight is recorded (load_note).",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Exercise name, e.g. 'Back Squat'"},
            },
            "required": ["name"],
        },
    },
}

In [7]:
# Example
print(get_exercise_info("Back Squat"))

[{"name":"Back Squat","primary_muscle":"quads","secondary_muscles":"glutes;hamstrings","equipment":"barbell","load_note":"total bar weight"}]


### Tool 3: `calculator` - exact maths (from Week 4 lab)

LLMs are unreliable at arithmetic (numbers are split into tokens), so calculations such as
training volume = sets × reps × weight go to this tool.

In [8]:
def calculator(expression: str) -> str:
    """Safely evaluate a math expression."""
    allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    allowed.update({"abs": abs, "round": round, "sum": sum, "min": min, "max": max})
    try:
        return str(eval(expression, {"__builtins__": {}}, allowed))
    except Exception as e:
        return f"Error: {e}"


calculator_schema = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate a mathematical expression. Use for any arithmetic, e.g. training "
                       "volume = sets * reps * weight. Example: '3*8*60 + 3*8*50'.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "A valid Python math expression"},
            },
            "required": ["expression"],
        },
    },
}

In [9]:
# Example: bench press volume on 3 Nov (3 sets x 8 reps x 60 kg)
print(calculator("3*8*60"))

1440


### Tool 4: `search_guides` - RAG over the training guides

1. **Index:** each guide file is short, so one file = one chunk. Chroma stores an embedding of each.
2. **Retrieve:** the query is embedded with the same model and the most similar guides are returned.
3. **Generate:** happens in the agent loop, where the model reads the returned text and answers.

The first run downloads the embedding model `all-MiniLM-L6-v2` (~90 MB).

In [10]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

GUIDES = []
for f in sorted((DATA_DIR / "guides").glob("*.md")):
    text = f.read_text(encoding="utf-8")
    GUIDES.append({"id": f.stem, "title": text.splitlines()[0].lstrip("# "), "text": text})

emb_fn     = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma     = chromadb.Client()
guides_col = chroma.get_or_create_collection("training_guides", embedding_function=emb_fn,
                                             metadata={"hnsw:space": "cosine"})
if guides_col.count() == 0:
    guides_col.add(documents=[g["text"] for g in GUIDES],
                   metadatas=[{"title": g["title"]} for g in GUIDES],
                   ids=[g["id"] for g in GUIDES])
print(f"Indexed {guides_col.count()} guides")


def search_guides(query: str, k: int = 2) -> str:
    """Retrieve the k guideline documents most similar in meaning to the query."""
    res = guides_col.query(query_texts=[query], n_results=k)
    hits = [{"source": m["title"], "text": d}
            for d, m in zip(res["documents"][0], res["metadatas"][0])]
    return json.dumps(hits, indent=2)


search_guides_schema = {
    "type": "function",
    "function": {
        "name": "search_guides",
        "description": "Search the training guideline documents for rules and recommendations about "
                       "rest and recovery, weekly volume, progressive overload, deloads, warm-ups, "
                       "and combining badminton with gym training.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The training question or topic to search for"},
            },
            "required": ["query"],
        },
    },
}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 6 guides


In [11]:
# Example
print(search_guides("how many sets per muscle per week"))

[
  {
    "source": "Weekly Training Volume",
    "text": "# Weekly Training Volume\nVolume load for one exercise is calculated as sets x reps x weight (kg). Bodyweight exercises such as pull-ups are counted by sets and reps only.\nFor muscle growth, aim for 10 to 20 hard sets per primary muscle group per week. Fewer than 10 sets per week is considered low volume. More than 20 sets per week increases fatigue with little extra benefit for most people.\n"
  },
  {
    "source": "Warm-Up",
    "text": "# Warm-Up\nBefore lifting, do 5 to 10 minutes of light cardio, then 2 to 3 warm-up sets of the first exercise at about 40%, 60% and 80% of the working weight.\nWarm-up sets are not counted as working sets when calculating weekly volume.\n"
  }
]


### Tool registry (from Week 4 lab)

Maps each tool name (what the model outputs) to the function that runs it.

In [12]:
ALL_TOOL_SCHEMAS = [get_workouts_schema, get_exercise_info_schema, calculator_schema, search_guides_schema]

TOOL_REGISTRY = {
    "get_workouts":      get_workouts,
    "get_exercise_info": get_exercise_info,
    "calculator":        calculator,
    "search_guides":     search_guides,
}

def execute_tool(name: str, args: dict) -> str:
    if name not in TOOL_REGISTRY:
        return f"Error: unknown tool '{name}'"
    try:
        return TOOL_REGISTRY[name](**args)
    except Exception as e:
        return f"Tool error: {e}"

print(execute_tool("calculator", {"expression": "2 ** 8"}))

256


## 4. Agent loop

### 4a. `llm_call` - one request to the model (from Week 4 lab)

Sends the conversation (`messages`) and the tool schemas to Mango via the Orion API.
Mango replies with either a text answer or one or more tool calls.
The reply is normalised into a simple dict: `content`, `tool_calls`, `tokens` and `reasoning` (the last two for logging only).

In [13]:
from openai import OpenAI

client = OpenAI(api_key=ORION_API_KEY, base_url=ORION_BASE_URL)


def llm_call(messages: list, tools: list = None) -> dict:
    """Send messages (and optional tool schemas) to Mango and return a normalised reply."""
    kwargs = dict(model=ORION_MODEL, messages=messages, max_tokens=1024, temperature=0.1)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"

    resp = client.chat.completions.create(**kwargs)
    msg  = resp.choices[0].message

    return {
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": [
            {"id": tc.id, "name": tc.function.name, "args": json.loads(tc.function.arguments)}
            for tc in (msg.tool_calls or [])
        ],
        "tokens": resp.usage.total_tokens if resp.usage else None,
        "reasoning": getattr(msg, "reasoning_content", None),   # logged only, not shown to the user
    }

In [14]:
# Connection test: plain question, no tools
reply = llm_call([{"role": "user", "content": "Who are you? Answer in one sentence."}])
print("Reply: ", reply["content"])
print("Tokens:", reply["tokens"])

Reply:  

I am an AI digital assistant designed to provide information and answer questions.
Tokens: 63


In [15]:
# Tool-call test: does Mango choose a tool when tools are offered?
reply = llm_call(
    [{"role": "user", "content": "What did I do at the gym on 2026-11-03?"}],
    tools=ALL_TOOL_SCHEMAS,
)
print("Text:      ", repr(reply["content"]))
print("Tool calls:", reply["tool_calls"])

Text:       ''
Tool calls: [{'id': 'chatcmpl-tool-a0cdee7a6e299563', 'name': 'get_workouts', 'args': {'start_date': '2026-11-03', 'end_date': '2026-11-03'}}]


### 4b. `run_agent` - the agent loop (from Week 4 lab)

Each step:
1. Send the conversation and tool schemas to Mango (`llm_call`).
2. If Mango asks for tools, run each one (`execute_tool`), append the results to the conversation, and repeat.
3. If Mango replies with text, that is the final answer.
4. Stop after `max_steps` in any case.

The function returns a trace (tool calls, steps, tokens) for the failure log.
The system prompt is deliberately plain: no reasoning pattern and no extra rules.

In [16]:
SYSTEM_PROMPT = """You are a fitness assistant for one user.
You have tools to look up the user's workout log, look up exercise information, do calculations,
and search training guidelines.
Always use the appropriate tool to get accurate information before answering.
When you have all the information you need, give a clear final answer."""


def run_agent(user_query: str, max_steps: int = 8, verbose: bool = True) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_query},
    ]
    trace = {"query": user_query, "tool_calls": [], "steps": 0, "tokens": 0,
             "answer": None, "final_reasoning": None}

    if verbose:
        print(f"USER: {user_query}\n" + "-" * 60)

    for step in range(max_steps):
        response = llm_call(messages, tools=ALL_TOOL_SCHEMAS)
        trace["steps"] += 1
        trace["tokens"] += response["tokens"] or 0

        if response["tool_calls"]:
            # Record Mango's tool request in the conversation
            messages.append({
                "role": "assistant",
                "content": response["content"],
                "tool_calls": [
                    {"id": tc["id"], "type": "function",
                     "function": {"name": tc["name"], "arguments": json.dumps(tc["args"])}}
                    for tc in response["tool_calls"]
                ],
            })
            # Run each tool and send the result back
            for tc in response["tool_calls"]:
                result = execute_tool(tc["name"], tc["args"])
                trace["tool_calls"].append({"name": tc["name"], "args": tc["args"]})
                if verbose:
                    print(f"[Step {step+1}] TOOL: {tc['name']}({tc['args']})")
                    print(f"         RESULT: {result[:300]}{'...' if len(result) > 300 else ''}")
                messages.append({"role": "tool", "tool_call_id": tc["id"],
                                 "name": tc["name"], "content": result})
        else:
            trace["answer"] = response["content"]
            trace["final_reasoning"] = response["reasoning"]
            break
    else:
        trace["answer"] = "Max steps reached without a final answer."

    if verbose:
        print("-" * 60 + f"\nAGENT: {trace['answer']}")
        if not (trace["answer"] or "").strip():
            print(f"(empty answer; reasoning_content was: {trace['final_reasoning']})")
        print(f"\n(steps: {trace['steps']}, tool calls: {len(trace['tool_calls'])}, tokens: {trace['tokens']})")
    return trace

In [17]:
# Example run with an explicit date
result = run_agent("What did I do at the gym on 2026-11-03?")

USER: What did I do at the gym on 2026-11-03?
------------------------------------------------------------
[Step 1] TOOL: get_workouts({'start_date': '2026-11-03', 'end_date': '2026-11-03'})
         RESULT: [{"date":"2026-11-03","day":"Tue","session":"Upper A","exercise":"Bench Press","sets":3,"reps":8,"weight_kg":60.0,"duration_min":null,"notes":null},{"date":"2026-11-03","day":"Tue","session":"Upper A","exercise":"Barbell Row","sets":3,"reps":8,"weight_kg":50.0,"duration_min":null,"notes":null},{"dat...
[Step 2] TOOL: calculator({'expression': '3*8*60'})
         RESULT: 1440
[Step 2] TOOL: calculator({'expression': '3*8*50'})
         RESULT: 1200
[Step 2] TOOL: calculator({'expression': '3*8*35'})
         RESULT: 840
[Step 2] TOOL: calculator({'expression': '3*10*45'})
         RESULT: 1350
[Step 2] TOOL: calculator({'expression': '3*12*10'})
         RESULT: 360
------------------------------------------------------------
AGENT: 

On 2026-11-03, you completed an **Upper A** sess

## 5. Test queries (failure log)

Five queries from simple (1 tool call) to complex (3+ tool calls with dependencies).
The expected answers were calculated directly from the data, so each agent answer can be checked.

| # | Level | Expected tools | Expected answer |
|---|---|---|---|
| 1 | Simple | `get_exercise_info` | Quads (secondary: glutes) |
| 2 | Medium | `get_workouts` → `calculator` | 1,560 min = 26 hours (one session was 120 min, not 180) |
| 3 | Complex | `get_workouts` ×2 → `get_exercise_info` → `calculator` | Chest volume dropped from 2,040 kg (Bench 1,440 + Incline DB 600 per dumbbell; 2,640 kg counting both dumbbells) to 1,560 kg (Bench only; Upper B on 20 Nov was missed) |
| 4 | Complex + RAG | `get_workouts` → `search_guides` | 48 h between Thu and Sat lower sessions meets the rest rule; but Lower B on Saturday is the day before Sunday badminton (guide 06), and the routine has no planned rest day (guide 01); the only day without training, 20 Nov, was a missed session |
| 5 | Trap | `get_workouts` (+ `get_exercise_info`) | No conventional Deadlift is logged; only Romanian Deadlift (60 → 70 kg) |

### 5a. Ground truth: expected answers computed from the data

Calculates the correct answer to each test query with pandas (no LLM), so every judgement in the
failure log can be checked. It reads the CSV files itself and does not call the model.
This cell was added after the official run of the test queries.

In [1]:
import pandas as pd

log = pd.read_csv("data/workout_log.csv")
ex  = pd.read_csv("data/exercises.csv")

# Q1: main muscle of the Bulgarian Split Squat
q1 = ex[ex["name"] == "Bulgarian Split Squat"].iloc[0]
print(f"Q1: {q1['primary_muscle']} (secondary: {q1['secondary_muscles']})")

# Q2: total badminton hours
bad = log[log["exercise"] == "Badminton"]
print(f"Q2: {len(bad)} sessions, {bad['duration_min'].sum():.0f} min = {bad['duration_min'].sum() / 60:.1f} hours")

# Q3: chest volume per week (sets x reps x weight); dumbbell weight is recorded per dumbbell
m = log.merge(ex, left_on="exercise", right_on="name")
chest = m[m["primary_muscle"] == "chest"].copy()
chest["volume"] = chest["sets"] * chest["reps"] * chest["weight_kg"]
chest["volume_both_db"] = chest["volume"] * chest["equipment"].eq("dumbbell").map({True: 2, False: 1})
print("Q3:")
for start, end in [("2026-11-02", "2026-11-08"), ("2026-11-16", "2026-11-22")]:
    w = chest[(chest["date"] >= start) & (chest["date"] <= end)]
    print(f"    {start} to {end}: {w['exercise'].tolist()} -> "
          f"{w['volume'].sum():.0f} kg as recorded, {w['volume_both_db'].sum():.0f} kg counting both dumbbells")

# Q4: lower-body sessions vs badminton and rest days
days = log.groupby("date")["session"].first()
dates = pd.to_datetime(pd.Series(days.index))
lower = [d for d, s in days.items() if s.startswith("Lower")]
print("Q4:")
for d in lower:
    nxt = (pd.to_datetime(d) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    print(f"    {d} {days[d]:8s} -> next day: {days.get(nxt, 'no entry')}")
all_days = pd.date_range(dates.min(), dates.max()).strftime("%Y-%m-%d")
print(f"    rest days (no gym and no badminton): {[d for d in all_days if d not in days.index] or 'none'}")

# Q5: which deadlift rows exist?
dl = log[log["exercise"].str.contains("deadlift", case=False)]
print(f"Q5: conventional 'Deadlift' rows: {int((log['exercise'] == 'Deadlift').sum())}; "
      f"rows containing 'deadlift': {dl[['date', 'exercise', 'weight_kg']].values.tolist()}")

Q1: quads (secondary: glutes)
Q2: 9 sessions, 1560 min = 26.0 hours
Q3:
    2026-11-02 to 2026-11-08: ['Bench Press', 'Incline Dumbbell Press'] -> 2040 kg as recorded, 2640 kg counting both dumbbells
    2026-11-16 to 2026-11-22: ['Bench Press'] -> 1560 kg as recorded, 1560 kg counting both dumbbells
Q4:
    2026-11-05 Lower A  -> next day: Upper B
    2026-11-07 Lower B  -> next day: Badminton
    2026-11-12 Lower A  -> next day: Upper B
    2026-11-14 Lower B  -> next day: Badminton
    2026-11-19 Lower A  -> next day: no entry
    2026-11-21 Lower B  -> next day: Badminton
    rest days (no gym and no badminton): ['2026-11-20']
Q5: conventional 'Deadlift' rows: 0; rows containing 'deadlift': [['2026-11-05', 'Romanian Deadlift', 60.0], ['2026-11-12', 'Romanian Deadlift', 65.0], ['2026-11-19', 'Romanian Deadlift', 70.0]]


In [18]:
TEST_QUERIES = [
    "What muscle does the Bulgarian Split Squat mainly train?",
    "How many hours of badminton did I play in total?",
    "How did my chest training volume change from 2-8 Nov 2026 to 16-22 Nov 2026?",
    "According to the training guidelines, is my lower-body gym training well planned around my badminton sessions?",
    "How much did my deadlift improve from my first to my last session?",
]

traces = []
for i, q in enumerate(TEST_QUERIES, 1):
    print(f"\n{'=' * 70}\nQUERY {i}")
    traces.append(run_agent(q))


QUERY 1
USER: What muscle does the Bulgarian Split Squat mainly train?
------------------------------------------------------------
[Step 1] TOOL: get_exercise_info({'name': 'Bulgarian Split Squat'})
         RESULT: [{"name":"Bulgarian Split Squat","primary_muscle":"quads","secondary_muscles":"glutes","equipment":"dumbbell","load_note":"weight per dumbbell"}]
------------------------------------------------------------
AGENT: 
(empty answer; reasoning_content was: The Bulgarian Split Squat mainly trains the quads (primary muscle) and also works the glutes (secondary muscles).)

(steps: 2, tool calls: 1, tokens: 1676)

QUERY 2
USER: How many hours of badminton did I play in total?
------------------------------------------------------------
------------------------------------------------------------
AGENT: 

I need to check your workout log to calculate your total badminton hours. Could you please specify the date range you'd like me to check? For example, "last 3 months" or specific

In [19]:
# Summary of all runs + save the traces for the failure log
summary = pd.DataFrame([{
    "#": i,
    "tools (in order)": " -> ".join(tc["name"] for tc in t["tool_calls"]) or "(none)",
    "tool calls": len(t["tool_calls"]),
    "steps": t["steps"],
    "tokens": t["tokens"],
    "empty answer": not (t["answer"] or "").strip(),
} for i, t in enumerate(traces, 1)])

Path("logs").mkdir(exist_ok=True)
with open("logs/midterm_traces.json", "w", encoding="utf-8") as f:
    json.dump(traces, f, indent=2, ensure_ascii=False)

summary

,#,tools (in order),tool calls,steps,tokens,empty answer
0,1,get_exercise_info,1,2,1676,True
1,2,(none),0,1,1104,False
2,3,get_workouts -> get_workouts -> get_exercise_i...,6,4,10719,False
3,4,search_guides,1,2,2321,False
4,5,get_workouts -> get_exercise_info -> get_workouts,3,4,4241,False


### 5b. Diagnosing empty answers (raw API reply)

Replays Query 1 and prints Mango's full raw message, to check where the final answer ends up (`content` vs `reasoning_content`).

In [20]:
# Diagnostic: raw API reply for Query 1 after the tool result (is the answer really empty?)
q = TEST_QUERIES[0]
msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}]

first = client.chat.completions.create(model=ORION_MODEL, messages=msgs, tools=ALL_TOOL_SCHEMAS,
                                       tool_choice="auto", max_tokens=1024, temperature=0.1)
m1 = first.choices[0].message
print("STEP 1 finish_reason:", first.choices[0].finish_reason)
print(json.dumps(m1.model_dump(), indent=2, ensure_ascii=False))

msgs.append({"role": "assistant", "content": m1.content or "",
             "tool_calls": [tc.model_dump() for tc in (m1.tool_calls or [])]})
for tc in (m1.tool_calls or []):
    msgs.append({"role": "tool", "tool_call_id": tc.id, "name": tc.function.name,
                 "content": execute_tool(tc.function.name, json.loads(tc.function.arguments))})

second = client.chat.completions.create(model=ORION_MODEL, messages=msgs, tools=ALL_TOOL_SCHEMAS,
                                        tool_choice="auto", max_tokens=1024, temperature=0.1)
print("\nSTEP 2 finish_reason:", second.choices[0].finish_reason)
print(json.dumps(second.choices[0].message.model_dump(), indent=2, ensure_ascii=False))
print("\nUsage:", second.usage)

STEP 1 finish_reason: tool_calls
{
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": null,
  "audio": null,
  "function_call": null,
  "tool_calls": [
    {
      "id": "chatcmpl-tool-bb01cf11ef64eb9a",
      "function": {
        "arguments": "{\"name\": \"Bulgarian Split Squat\"}",
        "name": "get_exercise_info"
      },
      "type": "function"
    }
  ],
  "reasoning_content": "The user is asking about the primary muscle group trained by the Bulgarian Split Squat exercise. I should use the get_exercise_info tool to look up this information.\n",
  "provider_specific_fields": {
    "reasoning": "The user is asking about the primary muscle group trained by the Bulgarian Split Squat exercise. I should use the get_exercise_info tool to look up this information.\n"
  }
}

STEP 2 finish_reason: stop
{
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": null,
  "audio": null,
  "function_call": null,
  "tool_calls": null,
  "reason